In [24]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
import json
import os

In [55]:
# ── CHANGE THESE TWO LINES per run ──────────────────────────────
MODEL_FOLDER = "muril-base-cased"
STATE        = "pretrained"
# ────────────────────────────────────────────────────────────────

BASE_DIR   = "/Users/harshaggarwal/Projects_4/hinemo_project/models/hidden_states"
MODEL_DIR  = f"{BASE_DIR}/{MODEL_FOLDER}"
OUTPUT_DIR = f"/Users/harshaggarwal/Projects_4/hinemo_project/models/emotion_classification_probing_results/{MODEL_FOLDER}"

os.makedirs(OUTPUT_DIR, exist_ok=True)

EMOTIONS = ["anger", "disgust", "joy", "sadness"]
layers   = list(range(1, 13))

print(f"Model:  {MODEL_FOLDER}")
print(f"State:  {STATE}")

Model:  muril-base-cased
State:  pretrained


In [56]:
hidden      = np.load(f"{MODEL_DIR}/hidden_{STATE}_full.npy")
meta        = pd.read_csv(f"{MODEL_DIR}/metadata_full.csv")
meta        = meta.reset_index(drop=True)
lambda_vals = meta["lambda"].values

le             = LabelEncoder()
emotion_labels = le.fit_transform(meta["gpt_emotion"].values)

n_samples, n_layers, hidden_dim = hidden.shape

print(f"Hidden states: {hidden.shape}")
print(f"Label mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")
print(f"\nEmotion distribution:")
print(meta["gpt_emotion"].value_counts())

# Pipeline: scale first, then classify
# Scaling inside pipeline means scaler is fit on train fold only — no leakage
def make_probe():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("clf",    LogisticRegression(max_iter=3000, random_state=524, C=1.0))
    ])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=524)

Hidden states: (4000, 12, 768)
Label mapping: {'anger': np.int64(0), 'disgust': np.int64(1), 'joy': np.int64(2), 'sadness': np.int64(3)}

Emotion distribution:
gpt_emotion
anger      1000
disgust    1000
joy        1000
sadness    1000
Name: count, dtype: int64


In [57]:
global_f1_per_layer = []

print("── GLOBAL EMOTION CLASSIFICATION PROBE ──\n")

for layer in range(n_layers):
    X = hidden[:, layer, :]
    y = emotion_labels

    scores  = cross_val_score(make_probe(), X, y, cv=cv, scoring="f1_macro")
    mean_f1 = scores.mean()

    global_f1_per_layer.append(mean_f1)
    print(f"Layer {layer+1:2d}: Macro F1 = {mean_f1:.4f} ± {scores.std():.4f}")

peak_layer = int(np.argmax(global_f1_per_layer)) + 1
print(f"\nPeak emotion classification layer: {peak_layer}")
print(f"Peak Macro F1: {global_f1_per_layer[peak_layer-1]:.4f}")

── GLOBAL EMOTION CLASSIFICATION PROBE ──

Layer  1: Macro F1 = 0.4795 ± 0.0247
Layer  2: Macro F1 = 0.4992 ± 0.0112
Layer  3: Macro F1 = 0.5059 ± 0.0223
Layer  4: Macro F1 = 0.5014 ± 0.0316
Layer  5: Macro F1 = 0.4947 ± 0.0082
Layer  6: Macro F1 = 0.5121 ± 0.0199
Layer  7: Macro F1 = 0.5272 ± 0.0103
Layer  8: Macro F1 = 0.5014 ± 0.0111
Layer  9: Macro F1 = 0.5332 ± 0.0106
Layer 10: Macro F1 = 0.5510 ± 0.0255
Layer 11: Macro F1 = 0.4926 ± 0.0150
Layer 12: Macro F1 = 0.5106 ± 0.0251

Peak emotion classification layer: 10
Peak Macro F1: 0.5510


In [58]:
binary_f1_per_emotion  = {}
peak_layer_per_emotion = {}

print("── PER EMOTION BINARY PROBE ──\n")

for emotion in EMOTIONS:
    binary_labels = (meta["gpt_emotion"].values == emotion).astype(int)

    f1_curve = []
    for layer in range(n_layers):
        X = hidden[:, layer, :]
        y = binary_labels

        # class_weight balanced because binary split is always imbalanced
        # 1000 target emotion vs 3000 others
        probe = Pipeline([
            ("scaler", StandardScaler()),
            ("clf",    LogisticRegression(max_iter=3000, random_state=524,
                                          C=1.0, class_weight="balanced"))
        ])

        scores = cross_val_score(probe, X, y, cv=cv, scoring="f1")
        f1_curve.append(scores.mean())

    peak = int(np.argmax(f1_curve)) + 1
    binary_f1_per_emotion[emotion]  = f1_curve
    peak_layer_per_emotion[emotion] = peak

    print(f"{emotion.upper():<10} peak layer: {peak}  "
          f"F1: {max(f1_curve):.4f}")

print(f"\nPeak layers per emotion: {peak_layer_per_emotion}")

── PER EMOTION BINARY PROBE ──

ANGER      peak layer: 10  F1: 0.5653
DISGUST    peak layer: 10  F1: 0.5242
JOY        peak layer: 9  F1: 0.6530
SADNESS    peak layer: 7  F1: 0.5511

Peak layers per emotion: {'anger': 10, 'disgust': 10, 'joy': 9, 'sadness': 7}


In [59]:
print("── λ-SPLIT EMOTION CLASSIFICATION PROBE ──\n")

low_mask  = lambda_vals < 0.33
high_mask = lambda_vals > 0.66

def get_balanced_subset(hidden, meta, mask, random_state=524):
    indices     = np.where(mask)[0]
    subset_meta = meta.iloc[indices].copy()
    subset_meta["_original_idx"] = indices

    min_class = subset_meta["gpt_emotion"].value_counts().min()

    sampled_parts = []
    for emotion in EMOTIONS:
        emotion_rows = subset_meta[subset_meta["gpt_emotion"] == emotion]
        sampled_parts.append(
            emotion_rows.sample(min_class, random_state=random_state)
        )

    balanced      = pd.concat(sampled_parts, ignore_index=True)
    final_indices = balanced["_original_idx"].values
    bucket_hidden = hidden[final_indices]
    bucket_labels = le.transform(balanced["gpt_emotion"].values)

    return bucket_hidden, bucket_labels, min_class

low_hidden,  low_labels,  low_n  = get_balanced_subset(hidden, meta, low_mask)
high_hidden, high_labels, high_n = get_balanced_subset(hidden, meta, high_mask)

print(f"Low-λ  bucket: {len(low_labels)} samples ({low_n} per class)")
print(f"High-λ bucket: {len(high_labels)} samples ({high_n} per class)")

results_by_bucket = {}

for bucket_name, bucket_hidden, bucket_labels in [
    ("low_lambda",  low_hidden,  low_labels),
    ("high_lambda", high_hidden, high_labels)
]:
    f1_curve = []
    print(f"\n{bucket_name.upper()}:")

    for layer in range(n_layers):
        X = bucket_hidden[:, layer, :]
        y = bucket_labels

        scores = cross_val_score(make_probe(), X, y, cv=cv, scoring="f1_macro")
        f1_curve.append(scores.mean())
        print(f"  Layer {layer+1:2d}: Macro F1 = {scores.mean():.4f}")

    peak = int(np.argmax(f1_curve)) + 1
    results_by_bucket[bucket_name] = {
        "f1_per_layer": f1_curve,
        "peak_layer"  : peak,
        "peak_f1"     : max(f1_curve)
    }
    print(f"  Peak layer: {peak}  F1: {max(f1_curve):.4f}")

print(f"\nLow-λ  peak layer: {results_by_bucket['low_lambda']['peak_layer']}")
print(f"High-λ peak layer: {results_by_bucket['high_lambda']['peak_layer']}")
print(f"Difference: {abs(results_by_bucket['low_lambda']['peak_layer'] - results_by_bucket['high_lambda']['peak_layer'])} layers")

── λ-SPLIT EMOTION CLASSIFICATION PROBE ──

Low-λ  bucket: 672 samples (168 per class)
High-λ bucket: 600 samples (150 per class)

LOW_LAMBDA:
  Layer  1: Macro F1 = 0.4335
  Layer  2: Macro F1 = 0.4582
  Layer  3: Macro F1 = 0.4704
  Layer  4: Macro F1 = 0.4844
  Layer  5: Macro F1 = 0.5048
  Layer  6: Macro F1 = 0.5304
  Layer  7: Macro F1 = 0.5197
  Layer  8: Macro F1 = 0.4922
  Layer  9: Macro F1 = 0.4909
  Layer 10: Macro F1 = 0.4518
  Layer 11: Macro F1 = 0.3922
  Layer 12: Macro F1 = 0.4762
  Peak layer: 6  F1: 0.5304

HIGH_LAMBDA:
  Layer  1: Macro F1 = 0.4183
  Layer  2: Macro F1 = 0.4274
  Layer  3: Macro F1 = 0.4324
  Layer  4: Macro F1 = 0.4648
  Layer  5: Macro F1 = 0.4683
  Layer  6: Macro F1 = 0.4577
  Layer  7: Macro F1 = 0.4720
  Layer  8: Macro F1 = 0.4993
  Layer  9: Macro F1 = 0.4673
  Layer 10: Macro F1 = 0.4748
  Layer 11: Macro F1 = 0.4306
  Layer 12: Macro F1 = 0.4839
  Peak layer: 8  F1: 0.4993

Low-λ  peak layer: 6
High-λ peak layer: 8
Difference: 2 layers


In [60]:
results = {
    "model"                  : MODEL_FOLDER,
    "state"                  : STATE,
    "global_f1_per_layer"    : global_f1_per_layer,
    "global_peak_layer"      : peak_layer,
    "binary_f1_per_emotion"  : binary_f1_per_emotion,
    "peak_layer_per_emotion" : peak_layer_per_emotion,
    "lambda_split"           : results_by_bucket
}

save_path = f"{OUTPUT_DIR}/emotion_classification_probe_{STATE}.json"
with open(save_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"Saved to {save_path}")

Saved to /Users/harshaggarwal/Projects_4/hinemo_project/models/emotion_classification_probing_results/muril-base-cased/emotion_classification_probe_pretrained.json


In [61]:
import json

BASE_PATH = "/Users/harshaggarwal/Projects_4/hinemo_project/models"
models = [
    "bert-base-multilingual-cased",
    "muril-base-cased",
    "xlm-roberta-base"
]
states   = ["pretrained", "finetuned"]
EMOTIONS = ["anger", "disgust", "joy", "sadness"]

print(f"{'Model+State':<25} {'Global Peak':<13} {'Peak F1':<10} {'Anger':<7} {'Disgust':<9} {'Joy':<7} {'Sadness':<9} {'Low-λ Peak':<12} {'High-λ Peak'}")
print("-" * 105)

for model in models:
    for state in states:
        path = f"{BASE_PATH}/phase7b_probing_results_emotion_classification_4k/{model}/emotion_classification_probe_{state}.json"
        with open(path) as f:
            r = json.load(f)

        label         = f"{model} {state}"
        global_peak   = r["global_peak_layer"]
        global_f1     = r["global_f1_per_layer"][global_peak - 1]
        emotion_peaks = [str(r["peak_layer_per_emotion"][e]) for e in EMOTIONS]
        low_peak      = r["lambda_split"]["low_lambda"]["peak_layer"]
        high_peak     = r["lambda_split"]["high_lambda"]["peak_layer"]

        print(f"{label:<25} {global_peak:<13} {global_f1:<10.4f} "
              f"{emotion_peaks[0]:<7} {emotion_peaks[1]:<9} "
              f"{emotion_peaks[2]:<7} {emotion_peaks[3]:<9} "
              f"{low_peak:<12} {high_peak}")

Model+State               Global Peak   Peak F1    Anger   Disgust   Joy     Sadness   Low-λ Peak   High-λ Peak
---------------------------------------------------------------------------------------------------------
bert-base-multilingual-cased pretrained 4             0.4931     1       4         3       3         5            1
bert-base-multilingual-cased finetuned 12            0.6151     12      12        12      12        10           11
muril-base-cased pretrained 10            0.5510     10      10        9       7         6            8
muril-base-cased finetuned 12            0.6729     12      10        12      12        12           11
xlm-roberta-base pretrained 6             0.6241     10      12        6       12        12           12
xlm-roberta-base finetuned 12            0.7613     12      12        12      12        12           12
